In [36]:
import subprocess
subprocess.run(['pip', 'install', 'rapidfuzz'], check=True)

CompletedProcess(args=['pip', 'install', 'rapidfuzz'], returncode=0)

In [37]:
import pandas as pd
import re
from rapidfuzz import fuzz, process

load data

In [38]:
market = pd.read_csv('Market PJMISO constraint list.csv')
dayzer = pd.read_csv('Dayzer PJMISO constraint list.csv')
pano = pd.read_csv('Pano PJMISO constraint list.csv')

print('Market shape:', market.shape)
print('Dayzer shape:', dayzer.shape)
print('Pano shape:', pano.shape)

print('\n--- Market columns ---\n')
print(market.head(5))

print('\n--- Dayzer columnes ---\n')
print(dayzer.head(5))

print('\n--- Pano columns ---\n')
print(pano.head(5))



Market shape: (5230, 6)
Dayzer shape: (13813, 2)
Pano shape: (21963, 5)

--- Market columns ---

                              CONSTRAINT                          CONTINGENCY  \
0  NOTTINGH 230 KV NOTTINGHM 2-3 SER DEV      L500.CONASTONE-PEACHBOTTOM.5012   
1   LENOX 115 KV LENOX-NMESHOPP NML 1090  L230.ETOWANDA-HILLSIDE.2002 [NYISO]   
2                EASTON  69 KV   EAS-EMU                               ACTUAL   
3                 SAYRECON230 KV SAY-SAY                               ACTUAL   
4     MOUN UGI 230 KV MOUN UGI 2 XFORMER          230/66.MOUNTAIN.T1 (SCTNLZ)   

    TOZONE                           REPORTEDNAME  CONSTRAINTID  CONTINGENCYID  
0      NaN  NOTTINGHM 2-3 SER DEV       A  230 KV   10002384305    10002493264  
1  PENELEC  LENOX-NMESHOPP NML 1090     B  115 KV   10000566138    10001865201  
2      DPL                EASTON  69 KV   EAS-EMU   10004072634    10000680485  
3     JCPL                SAYRECON230 KV  SAY-SAY   10001822928    10000680485  
4      NaN 

normalization:
1.lowercase
2.remove spaces
3.unifuy all separators into space
4.remove KV voltage labels to reduce noise

In [39]:
def normalize(s):
    if not isinstance(s,str):
        return ''
    s = s.lower()
    
    s = re.sub(r'[_\-\.]+', ' ', s)
    s = re.sub(r'\b\d+\s*kv\b', '', s)
    s = re.sub(r'\s+', ' ', s)
    s = s.strip()
    return s

# Quick test
print(normalize('NOTTINGH 230 KV NOTTINGHM 2-3 SER DEV'))
print(normalize('CHIC_AVE 138KV - PRAXAIR3 138KV (CHIC_AVE 138 KV CHI-PRA2)'))
print(normalize('L500.CONASTONE-PEACHBOTTOM.5012'))
    

nottingh nottinghm 2 3 ser dev
chic ave praxair3 (chic ave chi pra2)
l500 conastone peachbottom 5012


market kseys:

In [40]:
market['norm_constraint']= market['CONSTRAINT'].apply(normalize)
market['norm_contingency'] = market['CONTINGENCY'].apply(normalize)

market['market_constraint'] = market['CONSTRAINT'] + ' || ' + market['CONTINGENCY']
market['norm_key'] = market['norm_constraint'] + ' | ' + market['norm_contingency']

print(market[['CONSTRAINT', 'CONTINGENCY', 'norm_key']].head(5))

                              CONSTRAINT                          CONTINGENCY  \
0  NOTTINGH 230 KV NOTTINGHM 2-3 SER DEV      L500.CONASTONE-PEACHBOTTOM.5012   
1   LENOX 115 KV LENOX-NMESHOPP NML 1090  L230.ETOWANDA-HILLSIDE.2002 [NYISO]   
2                EASTON  69 KV   EAS-EMU                               ACTUAL   
3                 SAYRECON230 KV SAY-SAY                               ACTUAL   
4     MOUN UGI 230 KV MOUN UGI 2 XFORMER          230/66.MOUNTAIN.T1 (SCTNLZ)   

                                            norm_key  
0  nottingh nottinghm 2 3 ser dev | l500 conaston...  
1  lenox lenox nmeshopp nml 1090 | l230 etowanda ...  
2                            easton eas emu | actual  
3                    sayrecon230 kv say say | actual  
4  moun ugi moun ugi 2 xformer | 230/66 mountain ...  


panorama keys:

In [41]:
def extract_pano(s):
    if not isinstance(s,str):
        return ''
    match = re.search(r'\(([^)]+)\)',s)
    if match:
        return match.group(1).strip()
    return s.strip()

pano['facility_short'] = pano['Monitored Facility'].apply(extract_pano)
pano['norm_facility']  = pano['facility_short'].apply(normalize)
pano['norm_contingency'] = pano['Contingency Name'].apply(normalize)
pano['norm_key'] = pano['norm_facility'] + ' | ' + pano['norm_contingency']
pano['pano_constraint'] = pano['Monitored Facility'] + ' || ' + pano['Contingency Name']

print(pano[['Monitored Facility', 'facility_short', 'norm_key']].head(5))


        
    

                                  Monitored Facility  \
0  CHIC_AVE 138KV - PRAXAIR3 138KV (CHIC_AVE 138 ...   
1  177 BURN 345KV - MUNSTER2 345KV (177 BURN 345 ...   
2  CONASTON 500KV - CONASTON 230KV (CONASTON 500 ...   
3  SALTSPRG 138KV - MASURY 138KV (SALTSPRG 138 KV...   
4   BURMA 115KV - PINEY 115KV (BURMA 115 KV BUR-PIN)   

             facility_short                                           norm_key  
0  CHIC_AVE 138 KV CHI-PRA2  chic ave chi pra2 | l765 dumont wiltoncenter 1...  
1  177 BURN 345 KV BUR-MUN1  177 burn bur mun1 | l765 dumont wiltoncenter 1...  
2     CONASTON 500 KV 500-4      conaston 500 4 | l500 brighton conastone 5011  
3  SALTSPRG 138 KV SAL-MAS1            saltsprg sal mas1 | l345 niles shenango  
4      BURMA 115 KV BUR-PIN             burma bur pin | l230 glade warren 2088  


dayzer keys:

In [42]:
dayzer['norm_name'] = dayzer['NAME'].apply(normalize)
dayzer['dayzer_constraint'] = dayzer['NAME']

print(dayzer[['NAME', 'norm_name']].head(10))

                              NAME                      norm_name
0                Eastern Interface              eastern interface
1                Central Interface              central interface
2                Western Interface              western interface
3                  BCPEP Interface                bcpep interface
4  Black Oak - Bedington Interface  black oak bedington interface
5                PJMW500 Interface              pjmw500 interface
6         INTERFACE TOWANDA ACTUAL       interface towanda actual
7                W500-VP Interface              w500 vp interface
8               NORTH PE Interface             north pe interface
9   FALCONER_115 KV_FAL-WAR:ACTUAL        falconer fal war:actual


match market and panor

In [44]:
import rapidfuzz
print(rapidfuzz.__version__)

3.14.5


In [45]:
FUZZY_THRESHOLD = 85

pano_key_lookup = dict(zip(pano['norm_key'], pano['pano_constraint']))
pano_facility_lookup = dict(zip(pano['norm_facility'], pano['pano_constraint']))

pano_keys = list(pano_key_lookup.keys())
pano_facilities = list(pano_facility_lookup.keys())

market_to_pano = []

for _, row in market.iterrows():
    mkey = row['norm_key']
    mfac = row['norm_constraint']
    
  
    if mkey in pano_key_lookup:
        market_to_pano.append({
            'market_constraint': row['market_constraint'],
            'pano_constraint': pano_key_lookup[mkey],
            'pano_match_type': 'exact'
        })
        continue
    
    # fuzzy match

    result = process.extractOne(mfac, pano_facilities, scorer=fuzz.token_sort_ratio)
    if result and result[1] >= FUZZY_THRESHOLD:
        matched_facility = result[0]
        market_to_pano.append({
            'market_constraint': row['market_constraint'],
            'pano_constraint': pano_facility_lookup[matched_facility],
            'pano_match_type': f'fuzzy({result[1]})'
        })
    else:
        # if no match
        market_to_pano.append({
            'market_constraint': row['market_constraint'],
            'pano_constraint': None,
            'pano_match_type': 'no_match'
        })

market_pano_df = pd.DataFrame(market_to_pano)

print('Match summary (Market vs Pano):')
print(market_pano_df['pano_match_type'].value_counts())
print(market_pano_df.head(5))

Match summary (Market vs Pano):
pano_match_type
no_match                    2706
exact                        783
fuzzy(100.0)                 658
fuzzy(85.0)                  334
fuzzy(88.88888888888889)      89
                            ... 
fuzzy(89.28571428571429)       1
fuzzy(97.6)                    1
fuzzy(96.15384615384616)       1
fuzzy(97.5)                    1
fuzzy(96.0)                    1
Name: count, Length: 76, dtype: int64
                                   market_constraint  \
0  NOTTINGH 230 KV NOTTINGHM 2-3 SER DEV || L500....   
1  LENOX 115 KV LENOX-NMESHOPP NML 1090 || L230.E...   
2                  EASTON  69 KV   EAS-EMU || ACTUAL   
3                   SAYRECON230 KV SAY-SAY || ACTUAL   
4  MOUN UGI 230 KV MOUN UGI 2 XFORMER || 230/66.M...   

                                     pano_constraint           pano_match_type  
0                                               None                  no_match  
1  LENOX-NMESHOPP NML 1090     B  115 KV || L500....

match market and dayzer

In [50]:
dayzer_names = list(dayzer['norm_name'])
dayzer_lookup = dict(zip(dayzer['norm_name'], dayzer['dayzer_constraint']))

market_to_dayzer = []

for _, row in market.iterrows():
    mconstraint = row['norm_constraint']
    mcontingency = row['norm_contingency']
    mkey = row['norm_key']
    
    best_match = None
    best_score = 0
    best_type = 'no_match'
    
    # Try matching against constraint name
    r1 = process.extractOne(mconstraint, dayzer_names, scorer=fuzz.token_sort_ratio)
    if r1 and r1[1] > best_score:
        best_score = r1[1]
        best_match = r1[0]
        best_type = f'constraint_fuzzy({r1[1]})'
    
    # Try matching against contingency name
    r2 = process.extractOne(mcontingency, dayzer_names, scorer=fuzz.token_sort_ratio)
    if r2 and r2[1] > best_score:
        best_score = r2[1]
        best_match = r2[0]
        best_type = f'contingency_fuzzy({r2[1]})'
    
    if best_score >= FUZZY_THRESHOLD and best_match:
        market_to_dayzer.append({
            'market_constraint': row['market_constraint'],
            'dayzer_constraint': dayzer_lookup[best_match],
            'dayzer_match_type': best_type
        })
    else:
        market_to_dayzer.append({
            'market_constraint': row['market_constraint'],
            'dayzer_constraint': None,
            'dayzer_match_type': 'no_match'
        })

market_dayzer_df = pd.DataFrame(market_to_dayzer)

print('Match summary (Market vs Dayzer):')
print(market_dayzer_df['dayzer_match_type'].value_counts().head(10))
print(market_dayzer_df.head(5))
    
        

    
    

Match summary (Market vs Dayzer):
dayzer_match_type
no_match                                4979
contingency_fuzzy(85.0574712643678)       23
contingency_fuzzy(85.29411764705883)      22
contingency_fuzzy(85.71428571428572)      22
contingency_fuzzy(87.85046728971963)      22
contingency_fuzzy(87.71929824561404)      19
contingency_fuzzy(85.36585365853658)      16
contingency_fuzzy(86.66666666666667)      15
contingency_fuzzy(86.07594936708861)      14
contingency_fuzzy(87.27272727272728)      11
Name: count, dtype: int64
                                   market_constraint  \
0  NOTTINGH 230 KV NOTTINGHM 2-3 SER DEV || L500....   
1  LENOX 115 KV LENOX-NMESHOPP NML 1090 || L230.E...   
2                  EASTON  69 KV   EAS-EMU || ACTUAL   
3                   SAYRECON230 KV SAY-SAY || ACTUAL   
4  MOUN UGI 230 KV MOUN UGI 2 XFORMER || 230/66.M...   

                                   dayzer_constraint  \
0                                               None   
1  MAINESBG_345 KV_T1:L

combine results into the final table

In [51]:
final = market_pano_df[['market_constraint', 'pano_constraint', 'pano_match_type']].merge(
    market_dayzer_df[['market_constraint', 'dayzer_constraint', 'dayzer_match_type']],
    on='market_constraint',
    how='left'
)

print('Final table shape:', final.shape)
print(final.head(10))

Final table shape: (5230, 5)
                                   market_constraint  \
0  NOTTINGH 230 KV NOTTINGHM 2-3 SER DEV || L500....   
1  LENOX 115 KV LENOX-NMESHOPP NML 1090 || L230.E...   
2                  EASTON  69 KV   EAS-EMU || ACTUAL   
3                   SAYRECON230 KV SAY-SAY || ACTUAL   
4  MOUN UGI 230 KV MOUN UGI 2 XFORMER || 230/66.M...   
5  GRACETON 230 KV GRACETON-SAFEHARB 2303 || L500...   
6  94 HAURD-11323 11323 B 138 KV || L345.NELSON-E...   
7                    BERGEN 230 KV BER-HUD || ACTUAL   
8  LINE 69 KV MONR AE-VINELAND 0711-1 || L230.CUM...   
9  GARDNERS 115 KV GARDNERS-TEXSEAST GAR-TEX || L...   

                                     pano_constraint  \
0                                               None   
1  LENOX-NMESHOPP NML 1090     B  115 KV || L500....   
2  EASTON 69KV - EMUNI 69KV (EASTON 69 KV EAS-EMU...   
3                                               None   
4                                               None   
5  GRACETON-SAFEHA

export CSV

In [52]:
output = final[['market_constraint', 'dayzer_constraint', 'pano_constraint']].copy()
output.to_csv('matched_constraints.csv', index=False)
print('Saved to matched_constraints.csv')
print(output.head(10))

Saved to matched_constraints.csv
                                   market_constraint  \
0  NOTTINGH 230 KV NOTTINGHM 2-3 SER DEV || L500....   
1  LENOX 115 KV LENOX-NMESHOPP NML 1090 || L230.E...   
2                  EASTON  69 KV   EAS-EMU || ACTUAL   
3                   SAYRECON230 KV SAY-SAY || ACTUAL   
4  MOUN UGI 230 KV MOUN UGI 2 XFORMER || 230/66.M...   
5  GRACETON 230 KV GRACETON-SAFEHARB 2303 || L500...   
6  94 HAURD-11323 11323 B 138 KV || L345.NELSON-E...   
7                    BERGEN 230 KV BER-HUD || ACTUAL   
8  LINE 69 KV MONR AE-VINELAND 0711-1 || L230.CUM...   
9  GARDNERS 115 KV GARDNERS-TEXSEAST GAR-TEX || L...   

                                   dayzer_constraint  \
0                                               None   
1  MAINESBG_345 KV_T1:L230.ETowanda-Hillside.2002...   
2                                               None   
3                                               None   
4                                               None   
5             